# LLMs as Short-Answer Markers: Statistics and Figures

Reproduces every table, figure and number in the paper from `data/answers.csv` and the files in `marks/`. Run from the repository root, or open in Colab and run all cells; outputs are written to `results/`.

## Setup

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib as mpl
warnings.filterwarnings('ignore')

if not os.path.exists('data/answers.csv') and os.path.exists('../data/answers.csv'):
    os.chdir('..')
if not os.path.exists('data/answers.csv'):
    os.system('git clone -q https://github.com/sajeewa0825/llm-short-answer-marking.git')
    os.chdir('llm-short-answer-marking')

DATA_DIR = os.environ.get('DATA_DIR', '.')
OUT_DIR = os.environ.get('OUT_DIR', 'results')
os.makedirs(OUT_DIR, exist_ok=True)

B = 2000
NPERM = 10000
SEED = 42
MODELS = {
    'Claude Opus 5.5': 'marks/claude_opus_5.5.csv',
    'GPT-5.6': 'marks/gpt_5.6.csv',
    'Gemini 4 (High)': 'marks/gemini_4_high.csv',
    'Gemini 4 (Low)': 'marks/gemini_4_low.csv',
    'Kimi K3': 'marks/kimi_k3.csv',
    'Llama 4 Maverick': 'marks/llama_4_maverick.csv',
}
SCREENED = {
    'Gemini 3.1 Pro (High)': 'marks/excluded/gemini_3.1_pro_high.csv', 'Gemini 3.1 Pro (Low)': 'marks/excluded/gemini_3.1_pro_low.csv',
    'Gemini 3.5 Flash-Lite (High)': 'marks/excluded/gemini_3.5_flash_lite_high.csv', 'Gemini 3.5 Flash-Lite (Low)': 'marks/excluded/gemini_3.5_flash_lite_low.csv',
    'Gemini 3.8 Flash (High)': 'marks/excluded/gemini_3.8_flash_high.csv', 'Gemini 3.8 Flash (Low)': 'marks/excluded/gemini_3.8_flash_low.csv',
}
ORDER = list(MODELS)
DS = ['mohler', 'saf']
STEP = {'mohler': 0.5, 'saf': 0.125}
MAX = {'mohler': 5.0, 'saf': 1.0}
K = {ds: int(round(MAX[ds] / STEP[ds])) + 1 for ds in DS}
NAME = {'mohler': 'Mohler (0–5)', 'saf': 'SAF (0–1)'}

## Load human marks and LLM marks

In [ ]:
h = pd.read_csv(f'{DATA_DIR}/data/answers.csv')
EMPTY = re.compile(r'^(no response|no submission|empty (response|submission)|not answered)\.?$', re.I)
h['empty'] = h.student_answer.astype(str).str.strip().str.match(EMPTY)
h['name_err'] = h.student_answer.astype(str).str.strip().eq('#NAME?')
test_all = h[h.role == 'test'].copy()
test_noerr = test_all[~test_all.name_err].copy()
test = test_noerr[~test_noerr['empty']].copy()

def load(path):
    d = pd.read_csv(f'{DATA_DIR}/{path}')
    return d[['answer_id']].assign(guide_level=d.guidance_level, score=pd.to_numeric(d.score, errors='coerce'), reason=d.rationale.astype(str))

RAW = {name: load(p) for name, p in {**MODELS, **SCREENED}.items()}

def build_long(names, t):
    out = []
    for name in names:
        d = RAW[name].copy()
        d['model'] = name
        out.append(d)
    out = pd.concat(out).merge(t[['answer_id', 'dataset', 'question_id', 'human_score', 'human_1', 'human_2', 'max_score', 'empty']], on='answer_id')
    out['step'] = out.dataset.map(STEP)
    out['offgrid'] = np.abs(out.score / out.step - np.round(out.score / out.step)) > 1e-9
    out['score_r'] = (np.round(out.score / out.step) * out.step).clip(0, out.max_score)
    out['err'] = out.score_r - out.human_score
    out['abs_err'] = out.err.abs()
    return out.dropna(subset=['score'])

long = build_long(ORDER, test)
long_ne = build_long(ORDER, test_noerr)
long_all = build_long(ORDER, test_all)
pd.DataFrame({'sampled': test_all.groupby('dataset').size(), 'name_error': test_all.groupby('dataset').name_err.sum(),
              'non_response': test_noerr.groupby('dataset')['empty'].sum(), 'primary': test.groupby('dataset').size(),
              'questions': test.groupby('dataset').question_id.nunique()})

## Core statistics

In [ ]:
def qwk(a, b, ds, grid=None):
    st = STEP[ds] if grid is None else grid
    k = K[ds] if grid is None else int(round(MAX[ds] / grid)) + 1
    a = np.round(np.asarray(a, float) / st).astype(int)
    b = np.round(np.asarray(b, float) / st).astype(int)
    o = np.bincount(a * k + b, minlength=k * k).reshape(k, k).astype(float)
    n = o.sum()
    e = np.outer(o.sum(1), o.sum(0)) / n
    i = np.arange(k)
    w = (i[:, None] - i[None, :]) ** 2
    den = (w * e).sum()
    return 1.0 - (w * o).sum() / den if den > 0 else np.nan

def clusters(ids, qids):
    q = pd.Series(np.arange(len(ids)), index=pd.Index(qids))
    groups = [g.values for _, g in q.groupby(level=0)]
    return groups

def cboot(groups, fn, seed, b=B):
    r = np.random.default_rng(seed)
    vals = np.empty(b)
    for t in range(b):
        pick = r.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[j] for j in pick])
        vals[t] = fn(idx)
    vals = vals[~np.isnan(vals)]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)), vals

def iboot(n, fn, seed, b=B):
    r = np.random.default_rng(seed)
    vals = np.array([fn(r.integers(0, n, n)) for _ in range(b)])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def holm(p):
    p = np.asarray(p, float)
    o = np.argsort(p)
    adj = np.minimum(1, np.maximum.accumulate(p[o] * (len(p) - np.arange(len(p)))))
    out = np.empty_like(adj)
    out[o] = adj
    return out

def wide(frame, value='score_r'):
    w = frame.pivot_table(index='answer_id', columns=['model', 'guide_level'], values=value, aggfunc='first')
    meta = frame.groupby('answer_id')[['question_id', 'human_score', 'human_1', 'human_2', 'dataset', 'max_score']].first().loc[w.index]
    return w, meta

W, META = {}, {}
for ds in DS:
    W[ds], META[ds] = wide(long[long.dataset == ds])

## Agreement with the gold score (Table 3)

In [ ]:
rows = []
seed = SEED
for ds in DS:
    w, meta = W[ds], META[ds]
    for m in ORDER:
        for lv in [1, 2]:
            ok = w[(m, lv)].notna().values
            p, y, q = w[(m, lv)].values[ok], meta.human_score.values[ok], meta.question_id.values[ok]
            g = clusters(np.arange(len(p)), q)
            lo, hi, _ = cboot(g, lambda i: qwk(p[i], y[i], ds), seed)
            ilo, ihi = iboot(len(p), lambda i: qwk(p[i], y[i], ds), seed)
            blo, bhi, _ = cboot(g, lambda i: np.mean(p[i] - y[i]), seed + 1)
            d = p - y
            wb = stats.wilcoxon(d, zero_method='wilcox') if np.any(d != 0) else None
            seed += 2
            rows.append({'dataset': ds, 'level': lv, 'model': m, 'n': len(p), 'n_questions': len(g), 'QWK': qwk(p, y, ds), 'QWK_lo': lo, 'QWK_hi': hi,
                         'QWK_lo_answer': ilo, 'QWK_hi_answer': ihi, 'r': stats.pearsonr(p, y)[0], 'MAE': float(np.abs(d).mean()),
                         'exact': float(np.mean(np.isclose(p, y))), 'adjacent': float(np.mean(np.abs(d) <= STEP[ds] + 1e-9)),
                         'bias': float(d.mean()), 'bias_lo': blo, 'bias_hi': bhi, 'bias_p': float(wb.pvalue) if wb else 1.0,
                         'bias_W': float(wb.statistic) if wb else np.nan, 'bias_nonzero': int(np.sum(d != 0)),
                         'bias_pos': int(np.sum(d > 0)), 'bias_neg': int(np.sum(d < 0))})
main = pd.DataFrame(rows)
main['bias_p_holm'] = holm(main.bias_p.values)
main['ci_width_ratio'] = (main.QWK_hi - main.QWK_lo) / (main.QWK_hi_answer - main.QWK_lo_answer)
main.pivot_table(index='model', columns=['dataset', 'level'], values='QWK').loc[ORDER].round(3)

## Sample characteristics (Table 1)

In [ ]:
def describe(t, label):
    return pd.DataFrame([{'set': label, 'dataset': ds, 'answers': len(x), 'questions': x.question_id.nunique(),
                          'per_question_median': float(x.groupby('question_id').size().median()),
                          'mean_norm': x.human_score.mean() / MAX[ds], 'sd_norm': x.human_score.std() / MAX[ds],
                          'zero': int((x.human_score == 0).sum()), 'non_response': int(x['empty'].sum()), 'name_err': int(x.name_err.sum()),
                          'median_words': float(x.student_answer.astype(str).str.split().str.len().median())} for ds, x in t.groupby('dataset')])
pool = h[h.role != 'example']
bands = pd.cut(h.human_score / h.max_score, [-0.01, 0.25, 0.5, 0.75, 1.0], labels=['[0,.25]', '(.25,.5]', '(.5,.75]', '(.75,1]'])
alloc = pd.crosstab([h.dataset, h.role], bands, normalize='index').round(3)
desc = pd.concat([describe(test_all, 'sampled'), describe(test, 'primary')])
display(alloc)
desc

## Human ceiling and rater-substitution analysis (Table 4)

In [ ]:
wm, mm = W['mohler'], META['mohler']
r1, r2 = mm.human_1.values.astype(float), mm.human_2.values.astype(float)
gq = clusters(np.arange(len(r1)), mm.question_id.values)
qi = lambda a, b, idx=None: qwk(a if idx is None else a[idx], b if idx is None else b[idx], 'mohler', grid=1.0)
hh = {'QWK': qi(r1, r2), 'r': stats.pearsonr(r1, r2)[0], 'MAE': float(np.abs(r1 - r2).mean()), 'exact': float(np.mean(r1 == r2)), 'n': len(r1)}
hh['QWK_lo'], hh['QWK_hi'], _ = cboot(gq, lambda i: qi(r1, r2, i), 7)
sub = []
for lv in [1, 2]:
    for m in ORDER:
        p = wm[(m, lv)].values
        ok = ~np.isnan(p)
        pi = np.floor(p[ok] + 0.5)
        a1, a2 = r1[ok], r2[ok]
        g = clusters(np.arange(ok.sum()), mm.question_id.values[ok])
        d1 = lambda i: qi(pi, a1, i) - qi(a2, a1, i)
        d2 = lambda i: qi(pi, a2, i) - qi(a1, a2, i)
        l1, u1, _ = cboot(g, d1, 100 + lv)
        l2, u2, _ = cboot(g, d2, 200 + lv)
        sub.append({'level': lv, 'model': m, 'QWK_llm_r1': qi(pi, a1), 'QWK_llm_r2': qi(pi, a2), 'QWK_r1_r2': qi(a1, a2),
                    'diff_sub_r2': qi(pi, a1) - qi(a2, a1), 'diff_sub_r2_lo': l1, 'diff_sub_r2_hi': u1,
                    'diff_sub_r1': qi(pi, a2) - qi(a1, a2), 'diff_sub_r1_lo': l2, 'diff_sub_r1_hi': u2})
sub = pd.DataFrame(sub)
print({k: round(v, 3) for k, v in hh.items()})
sub.round(3)

## Effect of the reference answer (Table 5)

In [ ]:
lvl = []
for ds in DS:
    w, meta = W[ds], META[ds]
    for k_, m in enumerate(ORDER):
        ok = w[(m, 1)].notna().values & w[(m, 2)].notna().values
        p1, p2 = w[(m, 1)].values[ok], w[(m, 2)].values[ok]
        y, q = meta.human_score.values[ok], meta.question_id.values[ok]
        g = clusters(np.arange(len(y)), q)
        dq = lambda i: qwk(p2[i], y[i], ds) - qwk(p1[i], y[i], ds)
        lo, hi, _ = cboot(g, dq, 300 + k_ + (10 if ds == 'saf' else 0))
        obs = qwk(p2, y, ds) - qwk(p1, y, ds)
        rp = np.random.default_rng(400 + k_ + (10 if ds == 'saf' else 0))
        perm = np.empty(NPERM)
        for t in range(NPERM):
            s = rp.random(len(y)) < 0.5
            a, b = np.where(s, p1, p2), np.where(s, p2, p1)
            perm[t] = qwk(a, y, ds) - qwk(b, y, ds)
        pperm = (np.sum(np.abs(perm) >= abs(obs) - 1e-12) + 1) / (NPERM + 1)
        e1, e2 = np.abs(p1 - y), np.abs(p2 - y)
        mlo, mhi, _ = cboot(g, lambda i: np.mean(e2[i]) - np.mean(e1[i]), 500 + k_ + (10 if ds == 'saf' else 0))
        dif = e1 - e2
        nz = dif[dif != 0]
        rk = stats.rankdata(np.abs(nz))
        lvl.append({'dataset': ds, 'model': m, 'n': len(y), 'QWK_L1': qwk(p1, y, ds), 'QWK_L2': qwk(p2, y, ds), 'dQWK': obs, 'dQWK_lo': lo, 'dQWK_hi': hi,
                    'p_perm': pperm, 'dMAE': float(e2.mean() - e1.mean()), 'dMAE_lo': mlo, 'dMAE_hi': mhi,
                    'p_wilcoxon': float(stats.wilcoxon(e1, e2, zero_method='wilcox').pvalue) if len(nz) else 1.0,
                    'rank_biserial': float((rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum()) if len(nz) else 0.0,
                    'n_changed': int(np.sum(p1 != p2)), 'n_discordant': int(len(nz)), 'same_share': float(np.mean(p1 == p2))})
lvl = pd.DataFrame(lvl)
lvl['p_perm_holm'] = holm(lvl.p_perm.values)
lvl['p_wilcoxon_holm'] = holm(lvl.p_wilcoxon.values)
lvl.round(3)

## Install spaCy for the feature-based baseline

In [ ]:
import subprocess, sys
try:
    import spacy
    spacy.load('en_core_web_md')
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'spacy', 'https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl'], check=True)
    import spacy

## Feature-based baseline training

In [ ]:
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor

nlp = spacy.load('en_core_web_md', disable=['parser', 'ner', 'tagger', 'lemmatizer', 'attribute_ruler'])
tok = lambda s: set(re.findall(r'[a-z0-9]+', str(s).lower()))
nv = lambda v: v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
RIDGE, GBM = 'Ridge (feature-based)', 'Gradient boosting (feature-based)'
BPRED = {}
base_info = []
for ds in DS:
    d = h[(h.dataset == ds) & ~h.name_err].reset_index(drop=True)
    d['y'] = d.human_score / d.max_score
    sa, ra = d.student_answer.astype(str), d.reference_answer.astype(str)
    is_tr = (d.role != 'test').values
    wv = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(pd.concat([sa, ra]))
    cv = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True).fit(pd.concat([sa, ra]))
    Ws, Wr, Cs, Cr = wv.transform(sa), wv.transform(ra), cv.transform(sa), cv.transform(ra)
    vs = nv(np.vstack([x.vector for x in nlp.pipe(sa)]))
    vr = nv(np.vstack([x.vector for x in nlp.pipe(ra)]))
    F = pd.DataFrame({'cos_w': np.asarray(Ws.multiply(Wr).sum(1)).ravel(), 'cos_c': np.asarray(Cs.multiply(Cr).sum(1)).ravel(),
                      'cos_v': (vs * vr).sum(1),
                      'recall': [len(tok(a) & tok(b)) / max(1, len(tok(b))) for a, b in zip(sa, ra)],
                      'prec': [len(tok(a) & tok(b)) / max(1, len(tok(a))) for a, b in zip(sa, ra)],
                      'loglen': np.log1p(sa.str.split().str.len()),
                      'lenratio': sa.str.split().str.len() / ra.str.split().str.len().clip(lower=1)})
    knn, qm = np.zeros(len(d)), np.zeros(len(d))
    for q, g in d.groupby('question_id'):
        gtr = g[g.role != 'test']
        S = cosine_similarity(Cs[g.index], Cs[gtr.index])
        trpos = {i: k for k, i in enumerate(gtr.index)}
        for k, i in enumerate(g.index):
            s = S[k].copy()
            own = i in trpos
            if own:
                s[trpos[i]] = -1
            top = np.argsort(-s)[:5]
            knn[i] = np.average(gtr.y.values[top], weights=np.clip(s[top], 1e-3, None))
            qm[i] = (gtr.y.sum() - (d.y[i] if own else 0)) / max(1, len(gtr) - own)
    F['knn'], F['qmean'] = knn, qm
    st = STEP[ds] / MAX[ds]
    Xs = sparse.hstack([Ws, Cs, sparse.csr_matrix(F.values)]).tocsr()
    te = d[~is_tr]
    for bname, (mdl, X) in {RIDGE: (RidgeCV(alphas=[0.1, 0.3, 1, 3, 10]), Xs),
                            GBM: (HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42), F.values)}.items():
        mdl.fit(X[is_tr], d.y[is_tr])
        BPRED[(ds, bname)] = pd.Series(np.clip(np.round(mdl.predict(X[~is_tr]) / st) * st, 0, 1) * MAX[ds], index=te.answer_id.values)
    base_info.append({'dataset': ds, 'n_train': int(is_tr.sum()), 'n_train_anchor': int((d.role == 'example').sum())})
base_info = pd.DataFrame(base_info)
base_info

## Baseline agreement and paired differences (Table 3, Section 4.1)

In [ ]:
brow, pdiff = [], []
for ds in DS:
    w, meta = W[ds], META[ds]
    y, q = meta.human_score.values, meta.question_id.values
    g = clusters(np.arange(len(y)), q)
    for k_, bname in enumerate([RIDGE, GBM]):
        bp = BPRED[(ds, bname)].loc[w.index].values
        lo, hi, _ = cboot(g, lambda i: qwk(bp[i], y[i], ds), 600 + k_)
        brow.append({'dataset': ds, 'model': bname, 'n': len(y), 'QWK': qwk(bp, y, ds), 'QWK_lo': lo, 'QWK_hi': hi,
                     'r': stats.pearsonr(bp, y)[0], 'MAE': float(np.abs(bp - y).mean()), 'bias': float(np.mean(bp - y)),
                     'exact': float(np.mean(np.isclose(bp, y))), 'adjacent': float(np.mean(np.abs(bp - y) <= STEP[ds] + 1e-9))})
    bp = BPRED[(ds, RIDGE)].loc[w.index].values
    for lv in [1, 2]:
        for k_, m in enumerate(ORDER):
            p = w[(m, lv)].values
            ok = ~np.isnan(p)
            gg = clusters(np.arange(ok.sum()), q[ok])
            pp, yy, bb = p[ok], y[ok], bp[ok]
            lo, hi, v = cboot(gg, lambda i: qwk(pp[i], yy[i], ds) - qwk(bb[i], yy[i], ds), 700 + 10 * lv + k_ + (50 if ds == 'saf' else 0))
            pdiff.append({'dataset': ds, 'level': lv, 'model': m, 'diff_vs_ridge': qwk(pp, yy, ds) - qwk(bb, yy, ds), 'lo': lo, 'hi': hi})
baseline = pd.DataFrame(brow)
pdiff = pd.DataFrame(pdiff)
display(baseline.round(3))
pdiff.round(3)

## Exploratory comparison and median ensemble (Section 4.3)

In [ ]:
fried, posthoc, ens = [], [], []
for ds in DS:
    w, meta = W[ds], META[ds]
    y, q = meta.human_score.values, meta.question_id.values
    for lv in [1, 2]:
        sw = w.xs(lv, axis=1, level='guide_level')[ORDER]
        e = sw.sub(y, axis=0).abs().dropna()
        chi, p = stats.friedmanchisquare(*[e[c].values for c in ORDER])
        ranks = e.rank(axis=1).mean()
        fried.append({'dataset': ds, 'level': lv, 'n': len(e), 'chi2': chi, 'p': p, 'kendall_W': chi / (len(e) * (len(ORDER) - 1)),
                      'best': ranks.idxmin(), 'best_rank': ranks.min(), 'worst': ranks.idxmax(), 'worst_rank': ranks.max()})
        pp = []
        for i in range(len(ORDER)):
            for j in range(i + 1, len(ORDER)):
                a, b = e[ORDER[i]], e[ORDER[j]]
                t = stats.wilcoxon(a, b, zero_method='wilcox') if np.any(a != b) else None
                pp.append({'dataset': ds, 'level': lv, 'a': ORDER[i], 'b': ORDER[j], 'MAE_a': a.mean(), 'MAE_b': b.mean(), 'p': float(t.pvalue) if t else 1.0})
        pp = pd.DataFrame(pp)
        pp['p_holm'] = holm(pp.p.values)
        posthoc.append(pp)
        med = (np.round(sw.median(axis=1, skipna=True) / STEP[ds]) * STEP[ds]).values
        qs = {m: qwk(sw[m].dropna(), y[sw[m].notna().values], ds) for m in ORDER}
        bestm = max(qs, key=qs.get)
        bvals = sw[bestm].values
        ok = ~np.isnan(bvals)
        g = clusters(np.arange(ok.sum()), q[ok])
        mm_, yy, bb = med[ok], y[ok], bvals[ok]
        lo, hi, _ = cboot(g, lambda i: qwk(mm_[i], yy[i], ds), 800 + lv + (10 if ds == 'saf' else 0))
        dlo, dhi, _ = cboot(g, lambda i: qwk(mm_[i], yy[i], ds) - qwk(bb[i], yy[i], ds), 900 + lv + (10 if ds == 'saf' else 0))
        pair = [qwk(sw[[a, b]].dropna()[a], sw[[a, b]].dropna()[b], ds) for i, a in enumerate(ORDER) for b in ORDER[i + 1:]]
        ens.append({'dataset': ds, 'level': lv, 'QWK_ensemble': qwk(med, y, ds), 'lo': lo, 'hi': hi, 'best_single': bestm, 'best_QWK': qs[bestm],
                    'diff': qwk(mm_, yy, ds) - qwk(bb, yy, ds), 'diff_lo': dlo, 'diff_hi': dhi,
                    'inter_mean': float(np.mean(pair)), 'inter_min': float(np.min(pair)), 'inter_max': float(np.max(pair))})
fried, posthoc, ens = pd.DataFrame(fried), pd.concat(posthoc), pd.DataFrame(ens)
display(fried.round(3))
display(posthoc[posthoc.level == 2].round(3))
ens.round(3)

## Bias, limits of agreement and score bands (Table 6)

In [ ]:
ba = []
for ds in DS:
    for lv in [1, 2]:
        for m in ORDER:
            x = long[(long.dataset == ds) & (long.guide_level == lv) & (long.model == m)]
            d = x.err.values
            mean = ((x.score_r + x.human_score) / 2).values
            fit = stats.linregress(mean, d)
            res = d - (fit.intercept + fit.slope * mean)
            fa = stats.linregress(mean, np.abs(res))
            half = lambda v: 1.96 * np.sqrt(np.pi / 2) * (fa.intercept + fa.slope * v)
            lo_pt, hi_pt = 0.2 * MAX[ds], 0.8 * MAX[ds]
            ba.append({'dataset': ds, 'level': lv, 'model': m, 'bias': d.mean(), 'loa_lo': d.mean() - 1.96 * d.std(ddof=1), 'loa_hi': d.mean() + 1.96 * d.std(ddof=1),
                       'slope': fit.slope, 'slope_p': fit.pvalue, 'intercept': fit.intercept, 'abs_int': fa.intercept, 'abs_slope': fa.slope,
                       'reg_halfwidth_20': half(lo_pt), 'reg_halfwidth_80': half(hi_pt),
                       'reg_center_20': fit.intercept + fit.slope * lo_pt, 'reg_center_80': fit.intercept + fit.slope * hi_pt})
ba = pd.DataFrame(ba)
long['band'] = pd.cut(long.human_score / long.max_score, [-0.01, 0.0, 0.5, 0.99, 1.0], labels=['0', '(0,0.5]', '(0.5,1)', '1'])
band = long.groupby(['dataset', 'guide_level', 'model', 'band'], observed=True).agg(n=('err', 'size'), bias=('err', 'mean'), mae=('abs_err', 'mean')).reset_index()
band['bias_norm'] = band.bias / band.dataset.map(MAX)
band['mae_norm'] = band.mae / band.dataset.map(MAX)
display(ba[ba.level == 2].round(3))
band[band.guide_level == 2].pivot_table(index='model', columns=['dataset', 'band'], values='bias_norm', observed=True).round(3)

## Robustness analyses (Section 4.5)

In [ ]:
sens = []
for label, frame in [('with non-responses', long_ne), ('with non-responses and #NAME?', long_all)]:
    for (ds, lv, m), x in frame.groupby(['dataset', 'guide_level', 'model']):
        sens.append({'set': label, 'dataset': ds, 'level': lv, 'model': m, 'n': len(x), 'QWK': qwk(x.score_r, x.human_score, ds)})
sens = pd.DataFrame(sens)
kimi = long[(long.model == 'Kimi K3') & (long.dataset == 'saf')]
kimi_sens = pd.DataFrame([{'level': lv, 'n_offgrid': int(x.offgrid.sum()), 'QWK_all': qwk(x.score_r, x.human_score, 'saf'),
                           'QWK_ongrid_only': qwk(x[~x.offgrid].score_r, x[~x.offgrid].human_score, 'saf')} for lv, x in kimi.groupby('guide_level')])
long_scr = build_long(list(SCREENED), test)
screened_qwk = pd.DataFrame([{'dataset': ds, 'level': lv, 'model': m, 'n': len(x), 'QWK': qwk(x.score_r, x.human_score, ds)}
                             for (ds, lv, m), x in long_scr.groupby(['dataset', 'guide_level', 'model'])])
base_ne = []
for ds in DS:
    te = test_noerr[test_noerr.dataset == ds]
    for bname in [RIDGE, GBM]:
        bp = BPRED[(ds, bname)].loc[te.answer_id].values
        base_ne.append({'dataset': ds, 'model': bname, 'QWK_with_non_responses': qwk(bp, te.human_score.values, ds)})
base_ne = pd.DataFrame(base_ne)
display(sens.pivot_table(index='model', columns=['set', 'dataset', 'level'], values='QWK').loc[ORDER].round(3))
display(kimi_sens.round(3))
display(base_ne.round(3))
screened_qwk.pivot_table(index='model', columns=['dataset', 'level'], values='QWK').round(3)

## Output-integrity screening (Table 2)

In [ ]:
screen = []
for name in list(MODELS) + list(SCREENED):
    d = RAW[name].merge(test_all[['answer_id', 'dataset']], on='answer_id')
    mx = d.dataset.map(MAX)
    t = d.reason.str.replace(r"'[^']*'", "'X'", regex=True)
    wl = d.pivot_table(index='answer_id', columns='guide_level', values='score', aggfunc='first')
    screen.append({'model': name, 'marks': len(d), 'missing': int(d.score.isna().sum()),
                   'out_of_range': int(((d.score < 0) | (d.score > mx)).sum()),
                   'distinct': int(d.reason.nunique()), 'templates': int(t.nunique()),
                   'top5_share': float(t.value_counts().head(5).sum() / len(t)), 'same_L1_L2': float((wl[1] == wl[2]).mean())})
screen = pd.DataFrame(screen)
screen['rule_a'] = screen.templates < 10
screen['rule_b'] = screen.out_of_range > 0
screen['retained'] = ~(screen.rule_a | screen.rule_b)
examples = {name: RAW[name].reason.value_counts().head(3) for name in SCREENED}
offgrid_by = long_all.groupby(['model', 'dataset']).offgrid.sum().unstack().astype(int)
display(offgrid_by)
screen

## Save tables

In [ ]:
with pd.ExcelWriter(f'{OUT_DIR}/stats_tables_v3.xlsx') as xw:
    for nm, df in [('descriptives', desc), ('allocation', alloc.reset_index()), ('main_agreement', main), ('human_ceiling_sub', sub), ('level_effect', lvl),
                   ('baseline', baseline), ('baseline_info', base_info), ('llm_minus_ridge', pdiff), ('friedman', fried), ('posthoc', posthoc),
                   ('ensemble', ens), ('bland_altman', ba), ('score_band', band), ('sensitivity', sens), ('kimi_offgrid', kimi_sens),
                   ('baseline_with_nonresp', base_ne), ('screening', screen), ('screened_qwk', screened_qwk), ('offgrid_by', offgrid_by.reset_index())]:
        df.to_excel(xw, sheet_name=nm, index=False)
json.dump({'human_human': hh, 'n_marks_primary': int(len(long)), 'n_marks_all': int(len(long_all)), 'offgrid_total': int(long_all.offgrid.sum()),
           'examples': {k: v.to_dict() for k, v in examples.items()}}, open(f'{OUT_DIR}/extra_v3.json', 'w'), indent=1, default=float)
print('saved', OUT_DIR)

## Figure style

In [ ]:
COLORS = dict(zip(ORDER, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']))
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
mpl.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'], 'font.size': 7.5,
                     'axes.titlesize': 8, 'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.6, 'legend.frameon': False,
                     'savefig.dpi': 300, 'savefig.bbox': 'tight', 'figure.facecolor': 'white'})
FW = 4.8
minus = lambda v, f='+.3f': format(v, f).replace('-', '−')

def save(fig, name):
    fig.savefig(f'{OUT_DIR}/{name}.png')
    fig.savefig(f'{OUT_DIR}/{name}.pdf')
    plt.show()

## Supplementary Fig. S1: study design

In [ ]:
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(FW, 1.45))
ax.set_xlim(0, 100)
ax.set_ylim(0, 30)
ax.axis('off')
boxes = [(0.5, 'Data', 'Mohler 2,273\nSAF 2,110'),
         (20.5, 'Sample', '500 + 500\n935 substantive'),
         (40.5, 'Marking', 'L1, L2 prompts\n6 configurations'),
         (60.5, 'Screening', '12 runs, 6 kept\n11,219 marks'),
         (80.5, 'Analysis', 'agreement, effects,\nbias, ceiling')]
for x, head, body in boxes:
    ax.add_patch(FancyBboxPatch((x, 3), 17.5, 24, boxstyle='round,pad=0.3,rounding_size=1.5', fc='#f4f4f2', ec=MUTED, lw=0.7))
    ax.text(x + 8.75, 21.5, head, ha='center', va='center', fontsize=7, fontweight='bold', color=INK)
    ax.text(x + 8.75, 11, body, ha='center', va='center', fontsize=5.6, color=INK, linespacing=1.35)
for x in [18.4, 38.4, 58.4, 78.4]:
    ax.annotate('', xy=(x + 1.9, 15), xytext=(x - 0.1, 15), arrowprops=dict(arrowstyle='-|>', color=MUTED, lw=0.8))
save(fig, 'figS1_design')

## Fig. 1: QWK forest plot

In [ ]:
ypos = {m: i for i, m in enumerate(reversed(ORDER))}
fig, axes = plt.subplots(1, 2, figsize=(FW, 2.6), sharey=True)
for ax, ds in zip(axes, DS):
    if ds == 'mohler':
        ax.axvspan(hh['QWK_lo'], hh['QWK_hi'], color=GRID, zorder=0)
        ax.axvline(hh['QWK'], color=MUTED, ls='--', lw=0.9, zorder=1)
        ax.text(hh['QWK'] + 0.01, -1.55, 'R1–R2', ha='left', va='center', fontsize=6.5, color=MUTED)
    ax.axvline(0.70, color=MUTED, ls=':', lw=0.9, zorder=1)
    for m in ORDER:
        for lv, off, face in [(1, 0.17, 'white'), (2, -0.17, None)]:
            r = main[(main.dataset == ds) & (main.level == lv) & (main.model == m)].iloc[0]
            y = ypos[m] + off
            ax.plot([r.QWK_lo, r.QWK_hi], [y, y], color=COLORS[m], lw=1.2, solid_capstyle='round', zorder=2)
            ax.plot(r.QWK, y, 'o', ms=4, mfc=face or COLORS[m], mec=COLORS[m], mew=1.0, zorder=3)
    rb = baseline[(baseline.dataset == ds) & (baseline.model == RIDGE)].iloc[0]
    ax.plot([rb.QWK_lo, rb.QWK_hi], [-1, -1], color=MUTED, lw=1.2, solid_capstyle='round', zorder=2)
    ax.plot(rb.QWK, -1, 'D', ms=3.8, mfc=MUTED, mec=MUTED, zorder=3)
    ax.set_title(NAME[ds], color=INK, loc='left')
    ax.set_xlim(0.25, 1.0)
    ax.set_ylim(-1.9, len(ORDER) - 0.4)
    ax.set_xlabel('QWK with gold score')
    ax.grid(axis='x', color=GRID, lw=0.5)
    ax.set_axisbelow(True)
axes[0].set_yticks(list(ypos.values()) + [-1])
axes[0].set_yticklabels(list(ypos.keys()) + ['Ridge (feature-based)'])
axes[0].tick_params(axis='y', length=0)
handles = [plt.Line2D([], [], marker='o', ls='', mfc='white', mec=MUTED, mew=1.0, ms=4, label='L1'),
           plt.Line2D([], [], marker='o', ls='', mfc=MUTED, mec=MUTED, ms=4, label='L2'),
           plt.Line2D([], [], marker='D', ls='', mfc=MUTED, mec=MUTED, ms=3.8, label='Supervised'),
           plt.Line2D([], [], color=MUTED, ls=':', lw=0.9, label='0.70 criterion')]
fig.legend(handles=handles, loc='lower center', ncol=4, bbox_to_anchor=(0.55, -0.06))
fig.tight_layout()
save(fig, 'fig1_qwk_forest')

## Supplementary Fig. S2: reference-answer effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FW, 2.2), sharey=True)
for ax, ds in zip(axes, DS):
    ax.axvline(0, color=MUTED, lw=0.7)
    for m in ORDER:
        r = lvl[(lvl.dataset == ds) & (lvl.model == m)].iloc[0]
        sig = r.p_perm_holm < 0.05
        ax.plot([r.dQWK_lo, r.dQWK_hi], [ypos[m]] * 2, color=COLORS[m], lw=1.2, solid_capstyle='round')
        ax.plot(r.dQWK, ypos[m], 'o', ms=4.3, mfc=COLORS[m] if sig else 'white', mec=COLORS[m], mew=1.0)
        ax.text(0.99, ypos[m], minus(r.dQWK), transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=6.3, color=INK)
    ax.set_title(NAME[ds], color=INK, loc='left')
    ax.set_xlim(-0.25, 0.75)
    ax.set_xticks([-0.2, 0, 0.2, 0.4])
    ax.set_xlabel('ΔQWK (L2 − L1)')
    ax.grid(axis='x', color=GRID, lw=0.5)
    ax.set_axisbelow(True)
axes[0].set_yticks(list(ypos.values()))
axes[0].set_yticklabels(list(ypos.keys()))
axes[0].tick_params(axis='y', length=0)
fig.legend(handles=[plt.Line2D([], [], marker='o', ls='', mfc=MUTED, mec=MUTED, ms=4.3, label='Permutation p (Holm) < 0.05'),
                    plt.Line2D([], [], marker='o', ls='', mfc='white', mec=MUTED, mew=1.0, ms=4.3, label='Not significant')],
           loc='lower center', ncol=2, bbox_to_anchor=(0.55, -0.08))
fig.tight_layout()
save(fig, 'figS2_delta_qwk')

## Fig. 2: Bland–Altman plots with regression-based limits

In [ ]:
jit = np.random.default_rng(7)
fig, axes = plt.subplots(2, 3, figsize=(FW, 3.3), sharex=True, sharey=True)
xs = np.linspace(0, 5, 50)
for ax, m in zip(axes.ravel(), ORDER):
    x = long[(long.dataset == 'mohler') & (long.guide_level == 2) & (long.model == m)]
    mean = (x.score_r + x.human_score) / 2
    r = ba[(ba.dataset == 'mohler') & (ba.level == 2) & (ba.model == m)].iloc[0]
    ax.scatter(mean + jit.uniform(-0.08, 0.08, len(x)), x.err + jit.uniform(-0.08, 0.08, len(x)), s=3, color=COLORS[m], alpha=0.3, edgecolors='none')
    ax.axhline(0, color=GRID, lw=0.7)
    c = r.intercept + r.slope * xs
    hw = 1.96 * np.sqrt(np.pi / 2) * (r.abs_int + r.abs_slope * xs)
    ax.plot(xs, c, color=INK, lw=0.9)
    ax.plot(xs, c + hw, color=MUTED, lw=0.8, ls='--')
    ax.plot(xs, c - hw, color=MUTED, lw=0.8, ls='--')
    ax.text(0.03, 0.04, f'bias {minus(r.bias, "+.2f")}, slope {minus(r.slope, "+.2f")}', transform=ax.transAxes, fontsize=5.8, color=INK)
    ax.set_title(m, color=INK, fontsize=7)
    ax.set_xlim(-0.2, 5.2)
    ax.set_ylim(-5.2, 3.6)
for ax in axes[1]:
    ax.set_xlabel('Mean mark')
for ax in axes[:, 0]:
    ax.set_ylabel('LLM − human')
fig.tight_layout(h_pad=0.5, w_pad=0.5)
save(fig, 'fig2_bland_altman')

## Supplementary Fig. S3: score-band bias

In [ ]:
BANDS = ['0', '(0,0.5]', '(0.5,1)', '1']
LABELS = ['0', '(0, 0.5]', '(0.5, 1)', '1']
fig, axes = plt.subplots(1, 2, figsize=(FW, 2.4), sharey=True)
bw = 0.13
for ax, ds in zip(axes, DS):
    sb = band[(band.dataset == ds) & (band.guide_level == 2)]
    ax.axhline(0, color=MUTED, lw=0.7)
    for k, m in enumerate(ORDER):
        v = sb[sb.model == m].set_index('band').reindex(BANDS).bias_norm.values
        ax.bar(np.arange(4) + (k - 2.5) * bw, v, width=bw - 0.02, color=COLORS[m], label=m, zorder=2)
    n = sb.groupby('band', observed=True).n.max().reindex(BANDS).fillna(0).astype(int)
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'{l}\nn = {c}' for l, c in zip(LABELS, n)])
    ax.set_title(NAME[ds], color=INK, loc='left')
    ax.set_xlabel('Human score band (normalised)')
    ax.grid(axis='y', color=GRID, lw=0.5)
    ax.set_axisbelow(True)
axes[0].set_ylabel('Mean signed error (normalised)')
h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.17), fontsize=6.5)
fig.tight_layout()
save(fig, 'figS3_score_band')